# Customer Segmentation Using Machine Learning

## Project Overview
This notebook demonstrates a complete end-to-end Machine Learning project on **Customer Segmentation**. We will use the synthetic *Mall Customers* dataset which contains features like Age, Annual Income, and Spending Score. The goal is to segment customers into distinct behavioral groups to help a retail business optimize its marketing strategies.

### Objectives:
1. Perform **Exploratory Data Analysis (EDA)** to understand demographic and purchasing behavior correlations.
2. Standardize features and run **K-Means Clustering**.
3. Determine the optimal number of clusters using **Elbow Method** and **Silhouette Analysis**.
4. Apply **Dimensionality Reduction (PCA)** to visualize higher-dimensional clusters in a 2D space.
5. Profile clusters and assign **Business Personas** and **Marketing Strategies**.
6. Evaluate model performance against alternative algorithms like **Hierarchical Clustering** and **DBSCAN**.

## 1. Environment Setup & Data Loading

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root project folder to sys.path to access src modules
sys.path.append(os.path.abspath(".."))

from src.data_preprocessing import load_data, analyze_data_quality, preprocess_data
from src.clustering import (
    calculate_kmeans_metrics, fit_kmeans, fit_dbscan, 
    fit_hierarchical, run_pca, evaluate_model, analyze_clusters
)
from src.visualization import (
    plot_eda, plot_elbow, plot_kmeans_clusters, 
    plot_pca_clusters, plot_hierarchical_dendrogram
)
from main import assign_business_personas

print("Libraries successfully imported!")

In [ ]:
# Load dataset
data_path = "../data/Mall_Customers.csv"
df = load_data(data_path)
df.head()

## 2. Data Quality Analysis & Preprocessing

In [ ]:
# Check for missing values and duplicates
analyze_data_quality(df)

In [ ]:
# Preprocess and scale features
df_clean, scaled_df, scaler, gender_encoder = preprocess_data(df)
print("\nScaled features preview:")
scaled_df.head()

## 3. Exploratory Data Analysis (EDA)
Let's look at distributions, correlations, and feature relationships.

In [ ]:
# Plot numerical distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4A90E2', '#50E3C2', '#D0021B']
numeric_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

for idx, col in enumerate(numeric_cols):
    sns.histplot(df_clean[col], kde=True, ax=axes[idx], color=colors[idx], bins=20)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot relationship between Annual Income and Spending Score (Colored by Gender)
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Gender',
    data=df_clean,
    palette=['#F5A623', '#4A90E2'],
    s=80,
    edgecolor='black'
)
plt.title("Annual Income vs Spending Score by Gender", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(df_clean[numeric_cols + ['Gender_Encoded']].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap", fontweight='bold')
plt.show()

## 4. Clustering (K-Means)

### Finding the Optimal Clusters using Elbow and Silhouette Scores

In [ ]:
k_range, inertias, silhouette_scores = calculate_kmeans_metrics(scaled_df)

# Create the elbow plot
fig, ax1 = plt.subplots(figsize=(10, 5))
color = '#E74C3C'
ax1.set_xlabel('Number of Clusters (k)', fontweight='bold')
ax1.set_ylabel('Inertia', color=color, fontweight='bold')
ax1.plot(k_range, inertias, 'o-', color=color, linewidth=2, markersize=8)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#2ECC71'
ax2.set_ylabel('Silhouette Score', color=color, fontweight='bold')
ax2.plot(k_range, silhouette_scores, 's--', color=color, linewidth=2, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Optimal Clusters search: Elbow & Silhouette curves', fontweight='bold')
plt.show()

We can observe a clear bend (elbow) in the Inertia plot at **k=5**, and the Silhouette Score is also relatively high here. Thus, **k=5** is chosen as the optimal cluster size.

In [ ]:
# Fitting K-Means with K=5
optimal_k = 5
kmeans_model, kmeans_labels = fit_kmeans(scaled_df, n_clusters=optimal_k)
df_clean['KMeans_Cluster'] = kmeans_labels

# Evaluate K-Means Performance
kmeans_eval = evaluate_model(scaled_df, kmeans_labels)

### Plotting Clusters & Centroids

In [ ]:
# Calculate centroids in original space
centroids_orig = df_clean.groupby('KMeans_Cluster')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().values

plt.figure(figsize=(10, 7))
colors = ['#FF5A5F', '#3B5998', '#00A699', '#FC642D', '#767676']

for i in range(optimal_k):
    cluster_data = df_clean[df_clean['KMeans_Cluster'] == i]
    plt.scatter(cluster_data['Annual Income (k$)'], 
                cluster_data['Spending Score (1-100)'], 
                s=60, 
                color=colors[i], 
                label=f'Cluster {i}', 
                alpha=0.8, 
                edgecolors='black')

plt.scatter(centroids_orig[:, 0], centroids_orig[:, 1], s=250, color='yellow', marker='*', edgecolors='black', label='Centroids')
plt.title('Customer Segments (K-Means)', fontsize=15, fontweight='bold')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 5. PCA Dimensionality Reduction
Since our clustering happened in a 3D standardized space (Age, Income, Spending), PCA helps us project the clusters down into a 2D space to verify their separation.

In [ ]:
# Fit PCA
pca_model, pca_df = run_pca(scaled_df, n_components=2)
pca_df['Cluster'] = kmeans_labels

# Plot in PC space
plt.figure(figsize=(10, 7))
for i in range(optimal_k):
    cluster_data = pca_df[pca_df['Cluster'] == i]
    plt.scatter(cluster_data['PC1'], 
                cluster_data['PC2'], 
                s=60, 
                color=colors[i], 
                label=f'Cluster {i}', 
                alpha=0.8, 
                edgecolors='black')

plt.title('Clusters in PCA 2D Projected Space', fontsize=15, fontweight='bold')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 6. Business Profiling and Marketing Strategy
Let's look at the average stats of each cluster and label them with a business persona.

In [ ]:
# Profile each cluster
profile = analyze_clusters(df_clean, kmeans_labels)
profile = assign_business_personas(profile)

# Display the profile with persona and strategy
pd.set_option('display.max_colwidth', None)
profile

## 7. Model Comparison (Bonus Algorithms)
Let's evaluate K-Means against **Hierarchical Clustering** and **DBSCAN**.

In [ ]:
# Run Hierarchical (Ward)
hc_model, hc_labels = fit_hierarchical(scaled_df, n_clusters=optimal_k)
hc_eval = evaluate_model(scaled_df, hc_labels)

# Plot Dendrogram
plt.figure(figsize=(10, 5))
from scipy.cluster.hierarchy import dendrogram, linkage
linked = linkage(scaled_df, method='ward')
dendrogram(linked, no_labels=True)
plt.title('Hierarchical Clustering Dendrogram', fontweight='bold')
plt.xlabel('Customers')
plt.ylabel('Euclidean Distance')
plt.show()

In [ ]:
# Run DBSCAN
dbscan_model, dbscan_labels = fit_dbscan(scaled_df, eps=0.6, min_samples=4)
dbscan_eval = evaluate_model(scaled_df, dbscan_labels)

In [ ]:
# Model Comparison table
comparison_data = {
    'Algorithm': ['K-Means', 'Hierarchical (Ward)', 'DBSCAN'],
    'Clusters Found': [optimal_k, optimal_k, len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)],
    'Silhouette Score': [kmeans_eval['silhouette'], hc_eval['silhouette'], dbscan_eval['silhouette']],
    'Davies-Bouldin Index': [kmeans_eval['davies_bouldin'], hc_eval['davies_bouldin'], dbscan_eval['davies_bouldin']],
    'Calinski-Harabasz Score': [kmeans_eval['calinski_harabasz'], hc_eval['calinski_harabasz'], dbscan_eval['calinski_harabasz']]
}
comparison_df = pd.DataFrame(comparison_data)
comparison_df

### Conclusion and Key Insights
- **DBSCAN** achieves the highest Silhouette score of **~0.52**, identifying the core dense regions while successfully isolating outlier noise customers. However, DBSCAN makes it harder to categorize *every* customer as it marks some as noise.
- **K-Means** delivers a very strong and robust partition of the customer base into **5 segments**, which matches standard industry clusters. K-Means ensures every customer is grouped, allowing for clear product targeting.
- By profiling the K-Means clusters, marketing teams can deploy budget efficiently: directing high-end premium advertisements to the *Stars / Target Group*, value-oriented discount catalogs to the *Frugal / Conservative* group, and trendy flash-sale notifications to the *Spendthrifts*.